In [ ]:
pip install unsloth

In [ ]:
pip install transformers==4.56.2

In [ ]:
pip install --no-deps trl==0.22.2

In [ ]:
pip install jiwer

In [ ]:
pip install einops addict easydict

In [ ]:
pip install matplotlib

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install jiwer
!pip install einops addict easydict

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("unsloth/DeepSeek-OCR-2", local_dir = "deepseek_ocr2")

In [ ]:
from unsloth import FastVisionModel
import torch
from transformers import AutoModel
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = '0'

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Qwen3-VL-8B-Instruct-bnb-4bit",
    "unsloth/Qwen3-VL-8B-Thinking-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Instruct-bnb-4bit",
    "unsloth/Qwen3-VL-32B-Thinking-bnb-4bit",
]

model, tokenizer = FastVisionModel.from_pretrained(
    "./deepseek_ocr2",
    load_in_4bit=False,
    auto_model=AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=True,
    use_gradient_checkpointing="unsloth",
)

In [ ]:
# ==================== LOAD FROM HUGGING FACE ====================

from datasets import load_dataset
from huggingface_hub import login
import pandas as pd
from PIL import Image

# Login to Hugging Face (if private dataset)
login()  # Enter your token

In [ ]:
# Load your uploaded dataset
HF_USERNAME = "avishadilhara"
DATASET_NAME = "sinhala-ocr-lk-acts-1010"

print(f"Loading dataset from Hugging Face...")
dataset = load_dataset(f"{HF_USERNAME}/{DATASET_NAME}")

print(f"Dataset loaded successfully!")
print(f"   Train: {len(dataset['train'])} samples")
print(f"   Eval: {len(dataset['eval'])} samples")
print(f"   Test: {len(dataset['test'])} samples")

sample = dataset['train'][0]
print(f"\nSample structure:")
print(f"   Image size: {sample['image'].size}")
print(f"   Text length: {len(sample['text'])} chars")
print(f"   Year: {sample['year']}")
print(f"   Preview: {sample['text'][:100]}...")

In [ ]:
# Load dataset with transformation
from datasets import load_dataset
from PIL import Image

instruction = "<image>\nFree OCR."

def transform_sample(examples):
    """
    Transform function for dataset samples.
    Handles both single samples and batches.
    """
    images = examples['image']
    texts = examples['text']

    is_batch = isinstance(images, list)
    if not is_batch:
        images = [images]
        texts = [texts]

    messages_batch = []

    for img, text in zip(images, texts):
        try:
            if isinstance(img, list):
                img = img[0] if img else None

            if not isinstance(img, Image.Image):
                raise TypeError(f"Expected PIL Image, got {type(img)}")

            # Resize to max 1024 for memory efficiency
            max_size = 1024
            if img.width > max_size or img.height > max_size:
                ratio = min(max_size / img.width, max_size / img.height)
                new_size = (int(img.width * ratio), int(img.height * ratio))
                img = img.resize(new_size, Image.LANCZOS)

            msg = [
                {
                    "role": "<|User|>",
                    "content": instruction,
                    "images": [img]
                },
                {
                    "role": "<|Assistant|>",
                    "content": str(text)
                }
            ]
            messages_batch.append(msg)

        except Exception as e:
            print(f"Error processing sample: {e}")
            messages_batch.append([])

    if is_batch:
        return {"messages": messages_batch}
    else:
        return {"messages": messages_batch[0]}

print("Loading dataset...")
dataset = load_dataset("avishadilhara/sinhala-ocr-lk-acts-1010")

print(f"Loaded:")
print(f"   Train: {len(dataset['train'])} samples")
print(f"   Eval: {len(dataset['eval'])} samples")

print("\nApplying transform...")
dataset['train'].set_transform(transform_sample)
dataset['eval'].set_transform(transform_sample)

print(f"\nDatasets ready with correct format:")
print(f"   Instruction: '{instruction}'")
print(f"   Roles: '<|User|>' and '<|Assistant|>'")
print(f"   Train: {len(dataset['train'])} samples")
print(f"   Eval: {len(dataset['eval'])} samples")

# Verification
print("\nVerification:")

for split_name in ['train', 'eval']:
    try:
        sample = dataset[split_name][0]

        print(f"\n{split_name.upper()}:")
        print(f"   Sample keys: {list(sample.keys())}")
        print(f"   Messages count: {len(sample['messages'])}")

        if len(sample['messages']) == 2:
            msg0 = sample['messages'][0]
            msg1 = sample['messages'][1]

            print(f"   Correct! 2 messages")
            print(f"      Message 0:")
            print(f"         Type: {type(msg0)}")
            print(f"         Role: {msg0.get('role', 'N/A')}")
            print(f"         Content: {msg0.get('content', 'N/A')}")
            print(f"         Images: {len(msg0.get('images', []))} image(s)")
            print(f"      Message 1:")
            print(f"         Type: {type(msg1)}")
            print(f"         Role: {msg1.get('role', 'N/A')}")
            print(f"         Text length: {len(msg1.get('content', ''))} chars")
        else:
            print(f"   ERROR: Expected 2 messages, got {len(sample['messages'])}")
            for i, msg in enumerate(sample['messages']):
                print(f"      Message {i}: type={type(msg)}")

    except Exception as e:
        print(f"   Error accessing {split_name}: {e}")
        import traceback
        traceback.print_exc()

print("\nDataset ready for training!")

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
import torch
import math
from dataclasses import dataclass
from typing import Dict, List, Any, Tuple
from PIL import Image, ImageOps
from torch.nn.utils.rnn import pad_sequence
import io

from deepseek_ocr2.modeling_deepseekocr2 import (
    format_messages,
    text_encode,
    BasicImageTransform,
    dynamic_preprocess,
)

@dataclass
class DeepSeekOCR2DataCollator:
    """
    Data collator for DeepSeek OCR training.
    Args:
        tokenizer: Tokenizer
        model: Model
        image_size: Size for image patches (default: 768)
        base_size: Size for global view (default: 1024)
        crop_mode: Whether to use dynamic cropping for large images
        train_on_responses_only: If True, only train on assistant responses
    """
    tokenizer: Any
    model: Any
    image_size: int = 768
    base_size: int = 1024
    crop_mode: bool = True
    image_token_id: int = 128815
    train_on_responses_only: bool = True

    def __init__(
        self,
        tokenizer,
        model,
        image_size: int = 768,
        base_size: int = 1024,
        crop_mode: bool = True,
        train_on_responses_only: bool = True,
    ):
        self.tokenizer = tokenizer
        self.model = model
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.image_token_id = 128815
        self.dtype = model.dtype
        self.train_on_responses_only = train_on_responses_only

        self.image_transform = BasicImageTransform(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5),
            normalize=True
        )
        self.patch_size = 16
        self.downsample_ratio = 4

        if hasattr(tokenizer, 'bos_token_id') and tokenizer.bos_token_id is not None:
            self.bos_id = tokenizer.bos_token_id
        else:
            self.bos_id = 0
            print(f"Warning: tokenizer has no bos_token_id, using default: {self.bos_id}")

    def deserialize_image(self, image_data) -> Image.Image:
        """Convert image data to PIL Image in RGB mode"""
        if isinstance(image_data, Image.Image):
            return image_data.convert("RGB")
        elif isinstance(image_data, dict) and 'bytes' in image_data:
            image_bytes = image_data['bytes']
            image = Image.open(io.BytesIO(image_bytes))
            return image.convert("RGB")
        else:
            raise ValueError(f"Unsupported image format: {type(image_data)}")

    def calculate_image_token_count(self, image: Image.Image, crop_ratio: Tuple[int, int]) -> int:
        """Calculate the number of tokens this image will generate"""
        num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
        num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

        width_crop_num, height_crop_num = crop_ratio

        if self.crop_mode:
            img_tokens = num_queries_base * num_queries_base + 1
            if width_crop_num > 1 or height_crop_num > 1:
                img_tokens += (num_queries * width_crop_num) * (num_queries * height_crop_num)
        else:
            img_tokens = num_queries * num_queries + 1

        return img_tokens

    def process_image(self, image: Image.Image) -> Tuple[List, List, List, List, Tuple[int, int]]:
        """
        Process a single image based on crop_mode and size thresholds
        Returns: (images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio)
        """
        images_list = []
        images_crop_list = []
        images_spatial_crop = []

        if self.crop_mode:
            if image.size[0] <= 768 and image.size[1] <= 768:
                crop_ratio = (1, 1)
                images_crop_raw = []
            else:
                images_crop_raw, crop_ratio = dynamic_preprocess(
                    image, min_num=2, max_num=6,
                    image_size=self.image_size, use_thumbnail=False
                )

            global_view = ImageOps.pad(
                image, (self.base_size, self.base_size),
                color=tuple(int(x * 255) for x in self.image_transform.mean)
            )
            images_list.append(self.image_transform(global_view).to(self.dtype))

            width_crop_num, height_crop_num = crop_ratio
            images_spatial_crop.append([width_crop_num, height_crop_num])

            if width_crop_num > 1 or height_crop_num > 1:
                for crop_img in images_crop_raw:
                    images_crop_list.append(
                        self.image_transform(crop_img).to(self.dtype)
                    )

            num_queries = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
            num_queries_base = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)

            tokenized_image = ([self.image_token_id] * num_queries_base) * num_queries_base
            tokenized_image += [self.image_token_id]

            if width_crop_num > 1 or height_crop_num > 1:
                tokenized_image += ([self.image_token_id] * (num_queries * width_crop_num)) * (
                    num_queries * height_crop_num)

        else:
            crop_ratio = (1, 1)
            images_spatial_crop.append([1, 1])

            if self.base_size <= 768:
                resized_image = image.resize((self.base_size, self.base_size), Image.LANCZOS)
                images_list.append(self.image_transform(resized_image).to(self.dtype))
            else:
                global_view = ImageOps.pad(
                    image, (self.base_size, self.base_size),
                    color=tuple(int(x * 255) for x in self.image_transform.mean)
                )
                images_list.append(self.image_transform(global_view).to(self.dtype))

            num_queries = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tokenized_image = ([self.image_token_id] * num_queries) * num_queries
            tokenized_image += [self.image_token_id]

        return images_list, images_crop_list, images_spatial_crop, tokenized_image, crop_ratio

    def process_single_sample(self, messages: List[Dict]) -> Dict[str, Any]:
        """Process a single conversation into model inputs."""

        images = []
        for message in messages:
            if "images" in message and message["images"]:
                for img_data in message["images"]:
                    if img_data is not None:
                        pil_image = self.deserialize_image(img_data)
                        images.append(pil_image)

        if not images:
            raise ValueError("No images found in sample.")

        tokenized_str = []
        images_seq_mask = []
        images_list, images_crop_list, images_spatial_crop = [], [], []

        prompt_token_count = -1
        assistant_started = False
        image_idx = 0

        tokenized_str.append(self.bos_id)
        images_seq_mask.append(False)

        for message in messages:
            role = message["role"]
            content = message["content"]

            if role == "<|Assistant|>":
                if not assistant_started:
                    prompt_token_count = len(tokenized_str)
                    assistant_started = True

                content = f"{content.strip()} {self.tokenizer.eos_token}"

            text_splits = content.split('<image>')

            for i, text_sep in enumerate(text_splits):
                tokenized_sep = text_encode(self.tokenizer, text_sep, bos=False, eos=False)
                tokenized_str.extend(tokenized_sep)
                images_seq_mask.extend([False] * len(tokenized_sep))

                if i < len(text_splits) - 1:
                    if image_idx >= len(images):
                        raise ValueError(
                            f"Data mismatch: Found '<image>' token but no corresponding image."
                        )

                    image = images[image_idx]
                    img_list, crop_list, spatial_crop, tok_img, _ = self.process_image(image)

                    images_list.extend(img_list)
                    images_crop_list.extend(crop_list)
                    images_spatial_crop.extend(spatial_crop)

                    tokenized_str.extend(tok_img)
                    images_seq_mask.extend([True] * len(tok_img))

                    image_idx += 1

        if image_idx != len(images):
            raise ValueError(
                f"Data mismatch: Found {len(images)} images but only {image_idx} '<image>' tokens were used."
            )

        if not assistant_started:
            print("Warning: No assistant message found in sample. Masking all tokens.")
            prompt_token_count = len(tokenized_str)

        images_ori = torch.stack(images_list, dim=0)
        images_spatial_crop_tensor = torch.tensor(images_spatial_crop, dtype=torch.long)

        if images_crop_list:
            images_crop = torch.stack(images_crop_list, dim=0)
        else:
            images_crop = torch.zeros((1, 3, self.base_size, self.base_size), dtype=self.dtype)

        return {
            "input_ids": torch.tensor(tokenized_str, dtype=torch.long),
            "images_seq_mask": torch.tensor(images_seq_mask, dtype=torch.bool),
            "images_ori": images_ori,
            "images_crop": images_crop,
            "images_spatial_crop": images_spatial_crop_tensor,
            "prompt_token_count": prompt_token_count,
        }

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        """Collate batch of samples"""
        batch_data = []

        for feature in features:
            try:
                processed = self.process_single_sample(feature['messages'])
                batch_data.append(processed)
            except Exception as e:
                print(f"Error processing sample: {e}")
                continue

        if not batch_data:
            raise ValueError("No valid samples in batch")

        input_ids_list = [item['input_ids'] for item in batch_data]
        images_seq_mask_list = [item['images_seq_mask'] for item in batch_data]
        prompt_token_counts = [item['prompt_token_count'] for item in batch_data]

        input_ids = pad_sequence(input_ids_list, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        images_seq_mask = pad_sequence(images_seq_mask_list, batch_first=True, padding_value=False)

        labels = input_ids.clone()

        labels[labels == self.tokenizer.pad_token_id] = -100
        labels[images_seq_mask] = -100

        if self.train_on_responses_only:
            for idx, prompt_count in enumerate(prompt_token_counts):
                if prompt_count > 0:
                    labels[idx, :prompt_count] = -100

        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()

        images_batch = []
        for item in batch_data:
            images_batch.append((item['images_crop'], item['images_ori']))

        images_spatial_crop = torch.cat([item['images_spatial_crop'] for item in batch_data], dim=0)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "images": images_batch,
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": images_spatial_crop,
        }

In [ ]:
import os
import shutil
from transformers import TrainerCallback
from datetime import datetime

class AutoZipCheckpointCallback(TrainerCallback):
    """
    Automatically zips checkpoints and keeps only the latest N zip files.
    """

    def __init__(self, output_dir="./sinhala_ocr_outputs", keep_last_n=2, zip_format='zip'):
        """
        Args:
            output_dir: Training output directory
            keep_last_n: Number of latest zip files to keep
            zip_format: Archive format - 'zip'
        """
        self.output_dir = output_dir
        self.keep_last_n = keep_last_n
        self.zip_format = zip_format
        self.zipped_checkpoints = []

        self.zip_dir = os.path.join(output_dir, "checkpoint_zips")
        os.makedirs(self.zip_dir, exist_ok=True)

        print(f"AutoZipCheckpoint initialized:")
        print(f"   Output dir: {output_dir}")
        print(f"   Keeping latest: {keep_last_n} zips")
        print(f"   Zip storage: {self.zip_dir}")

    def on_save(self, args, state, control, **kwargs):
        """Called when a checkpoint is saved."""

        checkpoint_dir = f"checkpoint-{state.global_step}"
        checkpoint_path = os.path.join(self.output_dir, checkpoint_dir)

        if not os.path.exists(checkpoint_path):
            return

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        zip_name = f"checkpoint-{state.global_step}_step_{timestamp}"
        zip_path = os.path.join(self.zip_dir, zip_name)

        try:
            print(f"\nZipping checkpoint: {checkpoint_dir}...")
            shutil.make_archive(
                base_name=zip_path,
                format=self.zip_format,
                root_dir=self.output_dir,
                base_dir=checkpoint_dir
            )

            actual_zip = f"{zip_path}.zip"
            zip_size_mb = os.path.getsize(actual_zip) / (1024 * 1024)

            print(f"Zipped: {os.path.basename(actual_zip)} ({zip_size_mb:.1f} MB)")

            self.zipped_checkpoints.append({
                'path': actual_zip,
                'step': state.global_step,
                'timestamp': timestamp,
                'size_mb': zip_size_mb
            })

            self.zipped_checkpoints.sort(key=lambda x: x['step'], reverse=True)

            if len(self.zipped_checkpoints) > self.keep_last_n:
                old_zips = self.zipped_checkpoints[self.keep_last_n:]

                for old_zip in old_zips:
                    if os.path.exists(old_zip['path']):
                        print(f"Removing old zip: {os.path.basename(old_zip['path'])}")
                        os.remove(old_zip['path'])

                self.zipped_checkpoints = self.zipped_checkpoints[:self.keep_last_n]

            print(f"Current zips: {len(self.zipped_checkpoints)}/{self.keep_last_n}")
            for i, zip_info in enumerate(self.zipped_checkpoints, 1):
                print(f"   {i}. {os.path.basename(zip_info['path'])} ({zip_info['size_mb']:.1f} MB)")

        except Exception as e:
            print(f"Error zipping checkpoint: {e}")


auto_zip_callback = AutoZipCheckpointCallback(
    output_dir="./sinhala_ocr_outputs",
    keep_last_n=2,
    zip_format='zip'
)

print("\nAuto-zip callback ready!")

In [ ]:
torch.cuda.empty_cache()

In [ ]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, TrainerCallback
from unsloth import is_bf16_supported
import shutil
import os
import json

class SaveBestModelCallback(TrainerCallback):
    """Keeps a permanent copy of the best checkpoint"""
    def __init__(self, output_dir="./sinhala-ocr-outputs"):
        self.output_dir = output_dir
        self.best_model_dir = os.path.join(output_dir, "BEST_MODEL_BACKUP")
        self.best_metric = float("inf")
        self.pending_save = None

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        eval_loss = metrics.get("eval_loss")

        if eval_loss and eval_loss < self.best_metric:
            self.best_metric = eval_loss
            self.pending_save = {
                'step': state.global_step,
                'epoch': state.epoch,
                'eval_loss': eval_loss
            }

    def on_save(self, args, state, control, **kwargs):
        if self.pending_save and self.pending_save['step'] == state.global_step:
            latest_checkpoint = f"checkpoint-{state.global_step}"
            source_dir = os.path.join(self.output_dir, latest_checkpoint)

            if os.path.exists(source_dir):
                if os.path.exists(self.best_model_dir):
                    shutil.rmtree(self.best_model_dir)

                shutil.copytree(source_dir, self.best_model_dir)

                print("="*70)
                print("NEW BEST MODEL SAVED!")
                print(f"Epoch: {self.pending_save['epoch']:.2f}")
                print(f"Eval Loss: {self.pending_save['eval_loss']:.6f}")
                print(f"Location: {self.best_model_dir}")
                print("="*70)

                self.pending_save = None


class GPUCacheClearCallback(TrainerCallback):
    def __init__(self, clear_every_n_epochs=2):
        self.clear_every_n_epochs = clear_every_n_epochs
        self.last_cleared_epoch = -1

    def on_epoch_end(self, args, state, control, **kwargs):
        current_epoch = int(state.epoch)
        if current_epoch - self.last_cleared_epoch >= self.clear_every_n_epochs:
            torch.cuda.empty_cache()
            self.last_cleared_epoch = current_epoch

    def on_evaluate(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()

FastVisionModel.for_training(model)

data_collator = DeepSeekOCR2DataCollator(
    tokenizer=tokenizer,
    model=model,
    image_size=768,
    base_size=1024,
    crop_mode=True,
    train_on_responses_only=True,
)

training_args = TrainingArguments(
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    num_train_epochs=50,

    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_steps=10,

    optim="adamw_8bit",
    weight_decay=0.001,
    max_grad_norm=1.0,

    fp16=False,
    bf16=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    logging_steps=5,
    report_to="none",

    seed=3407,
    dataloader_num_workers=2,
    remove_unused_columns=False,

    output_dir="./sinhala_ocr_outputs",
)

best_model_saver = SaveBestModelCallback()

trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    data_collator=data_collator,
    train_dataset=dataset['train'],
    eval_dataset=dataset['eval'],
    args=training_args,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=5),
        GPUCacheClearCallback(clear_every_n_epochs=2),
        best_model_saver,
        auto_zip_callback,
    ]
)

print("Trainer configured!")
print(f"   Training samples: {len(dataset['train'])}")
print(f"   Validation samples: {len(dataset['eval'])}")
print(f"   Test samples: {len(dataset['test'])}")

In [ ]:
print("Starting training from start...")
trainer_stats = trainer.train()

In [ ]:
trainer_stats = trainer.train(resume_from_checkpoint="/kaggle/working/sinhala_ocr_outputs/checkpoint-267")
print("\nTraining completed!")
print(f"   Total steps: {trainer_stats.global_step}")
print(f"   Training time: {trainer_stats.metrics['train_runtime'] / 60:.1f} minutes")

In [ ]:
import os
import shutil
import json

print("="*80)
print("SAVING BEST MODEL FROM BEST_MODEL_BACKUP/")
print("="*80)

best_backup = "./sinhala_ocr_outputs/BEST_MODEL_BACKUP"
final_save_dir = "./sinhala_deepseek_ocr_lora_BEST"

if os.path.exists(best_backup):
    if os.path.exists(f"{best_backup}/trainer_state.json"):
        with open(f"{best_backup}/trainer_state.json", 'r') as f:
            state = json.load(f)
        print(f"Best Model Info:")
        print(f"   Epoch: {state['epoch']:.2f}")
        print(f"   Eval Loss: {state.get('best_metric', 'N/A'):.6f}")

    if os.path.exists(final_save_dir):
        shutil.rmtree(final_save_dir)
    shutil.copytree(best_backup, final_save_dir)

    model.save_pretrained("sinhala_deepseek_ocr_lora")
    tokenizer.save_pretrained("sinhala_deepseek_ocr_lora")

    print(f"Best model saved to: {final_save_dir}")
else:
    print("BEST_MODEL_BACKUP not found! Using memory fallback.")
    model.save_pretrained(final_save_dir)
    tokenizer.save_pretrained(final_save_dir)

print("="*80)

In [ ]:
from datetime import datetime

def zip_best_model(model_dir="./sinhala_deepseek_ocr_lora_BEST"):
    if not os.path.exists(model_dir):
        print(f"Error: {model_dir} not found!")
        return None

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_name = f"sinhala_ocr_BEST_{timestamp}"
    zip_path = os.path.join(".", zip_name)

    print(f"Zipping {model_dir}...")
    shutil.make_archive(
        base_name=zip_path,
        format='zip',
        root_dir=".",
        base_dir=os.path.basename(model_dir)
    )

    final_zip = f"{zip_path}.zip"
    zip_size_mb = os.path.getsize(final_zip) / (1024 * 1024)
    print(f"Created: {os.path.basename(final_zip)} ({zip_size_mb:.1f} MB)")

    return final_zip

zip_file = zip_best_model()
print(f"\nReady to download: {zip_file}")

In [ ]:
# Save LoRA adapters
model.save_pretrained("sinhala_deepseek_ocr_lora")
tokenizer.save_pretrained("sinhala_deepseek_ocr_lora")

print("Model saved locally!")

In [ ]:
import os
import shutil
from datetime import datetime

def zip_trained_model(
    model_dir="sinhala_deepseek_ocr_lora",
    output_dir="./",
    include_timestamp=True,
    also_zip_best_checkpoint=True,
    checkpoint_dir="./sinhala_ocr_outputs"
):
    """
    Zip the final trained model and optionally the best checkpoint.

    Args:
        model_dir: Directory containing the trained model
        output_dir: Where to save the zip files
        include_timestamp: Add timestamp to zip filename
        also_zip_best_checkpoint: Also zip the best checkpoint
        checkpoint_dir: Training output directory containing checkpoints
    """

    print("="*70)
    print("ZIPPING FINAL TRAINED MODEL")
    print("="*70)

    zipped_files = []

    # Zip trained model (LoRA adapters)
    if os.path.exists(model_dir):
        print(f"\nZipping trained model: {model_dir}/")

        model_size_mb = sum(
            os.path.getsize(os.path.join(dirpath, filename))
            for dirpath, dirnames, filenames in os.walk(model_dir)
            for filename in filenames
        ) / (1024 * 1024)

        print(f"   Original size: {model_size_mb:.1f} MB")

        if include_timestamp:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            zip_name = f"sinhala_ocr_lora_{timestamp}"
        else:
            zip_name = "sinhala_ocr_lora_final"

        zip_path = os.path.join(output_dir, zip_name)

        print(f"   Zipping to: {zip_name}.zip")
        shutil.make_archive(
            base_name=zip_path,
            format='zip',
            root_dir=os.path.dirname(model_dir) if os.path.dirname(model_dir) else '.',
            base_dir=os.path.basename(model_dir)
        )

        final_zip = f"{zip_path}.zip"
        zip_size_mb = os.path.getsize(final_zip) / (1024 * 1024)
        compression_ratio = (1 - zip_size_mb / model_size_mb) * 100

        print(f"   Created: {os.path.basename(final_zip)}")
        print(f"   Zip size: {zip_size_mb:.1f} MB")
        print(f"   Compression: {compression_ratio:.1f}% smaller")

        zipped_files.append({
            'name': os.path.basename(final_zip),
            'path': final_zip,
            'size_mb': zip_size_mb,
            'type': 'LoRA Model'
        })
    else:
        print(f"\nError: Model directory not found: {model_dir}")
        print("   Make sure you ran the model save cell first!")

    # Zip best checkpoint (optional)
    if also_zip_best_checkpoint:
        print(f"\nLooking for best checkpoint in: {checkpoint_dir}/")

        checkpoints = []
        if os.path.exists(checkpoint_dir):
            for item in os.listdir(checkpoint_dir):
                item_path = os.path.join(checkpoint_dir, item)
                if os.path.isdir(item_path) and item.startswith('checkpoint-'):
                    checkpoints.append(item_path)

        if checkpoints:
            checkpoints.sort(key=os.path.getmtime, reverse=True)
            best_checkpoint = checkpoints[0]

            print(f"   Found best checkpoint: {os.path.basename(best_checkpoint)}/")

            checkpoint_size_mb = sum(
                os.path.getsize(os.path.join(dirpath, filename))
                for dirpath, dirnames, filenames in os.walk(best_checkpoint)
                for filename in filenames
            ) / (1024 * 1024)

            print(f"   Original size: {checkpoint_size_mb:.1f} MB")

            if include_timestamp:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                checkpoint_zip_name = f"best_checkpoint_{timestamp}"
            else:
                checkpoint_zip_name = f"best_{os.path.basename(best_checkpoint)}"

            checkpoint_zip_path = os.path.join(output_dir, checkpoint_zip_name)

            print(f"   Zipping to: {checkpoint_zip_name}.zip")
            shutil.make_archive(
                base_name=checkpoint_zip_path,
                format='zip',
                root_dir=checkpoint_dir,
                base_dir=os.path.basename(best_checkpoint)
            )

            final_checkpoint_zip = f"{checkpoint_zip_path}.zip"
            checkpoint_zip_size_mb = os.path.getsize(final_checkpoint_zip) / (1024 * 1024)
            checkpoint_compression_ratio = (1 - checkpoint_zip_size_mb / checkpoint_size_mb) * 100

            print(f"   Created: {os.path.basename(final_checkpoint_zip)}")
            print(f"   Zip size: {checkpoint_zip_size_mb:.1f} MB")
            print(f"   Compression: {checkpoint_compression_ratio:.1f}% smaller")

            zipped_files.append({
                'name': os.path.basename(final_checkpoint_zip),
                'path': final_checkpoint_zip,
                'size_mb': checkpoint_zip_size_mb,
                'type': 'Best Checkpoint'
            })
        else:
            print(f"   No checkpoints found in {checkpoint_dir}")

    # Summary
    print("\n" + "="*70)
    print("ZIPPING COMPLETE")
    print("="*70)

    if zipped_files:
        print(f"\nCreated {len(zipped_files)} zip file(s):")
        total_size = 0

        for i, zf in enumerate(zipped_files, 1):
            print(f"\n{i}. {zf['name']}")
            print(f"   Type: {zf['type']}")
            print(f"   Size: {zf['size_mb']:.1f} MB")
            print(f"   Path: {zf['path']}")
            total_size += zf['size_mb']

        print(f"\nTotal size: {total_size:.1f} MB")

        print("\n" + "="*70)
        print("DOWNLOAD INSTRUCTIONS")
        print("="*70)

        print("\nKaggle:")
        print("   Right sidebar -> Output -> Click zip file to download")

        print("\nGoogle Colab:")
        print("   Run this to download:")
        for zf in zipped_files:
            print(f"   from google.colab import files")
            print(f"   files.download('{zf['path']}')")

        print("\nProgrammatic download:")
        for zf in zipped_files:
            print(f"   !cp {zf['path']} /kaggle/working/")

    else:
        print("\nNo files were zipped!")

    print("="*70)

    return zipped_files


zipped = zip_trained_model(
    model_dir="sinhala_deepseek_ocr_lora",
    output_dir="./",
    include_timestamp=True,
    also_zip_best_checkpoint=True,
    checkpoint_dir="./sinhala_ocr_outputs"
)

In [ ]:
# Quick test on a sample
from jiwer import cer
import os

FastVisionModel.for_training(model, False)

os.makedirs("./test_output", exist_ok=True)

test_sample = dataset['test'][58]

test_sample['image'].save("test_img.jpg")

print("="*80)
print("TESTING SINHALA OCR MODEL")
print("="*80)
print(f"\nRunning inference on test sample...")

prediction = model.infer(
    tokenizer,
    prompt="<image> OCR.",
    image_file="test_img.jpg",
    output_path="./test_output",
    base_size=1024,
    image_size=768,
    crop_mode=True,
    save_results=False,
)

print(f"\nInference Complete!\n")
print("="*80)
print("GROUND TRUTH (first 300 chars):")
print("="*80)
print(f"{test_sample['text'][:300]}...\n")

print("="*80)
print("PREDICTED (first 300 chars):")
print("="*80)
print(f"{prediction[:300]}...")

error_rate = cer(test_sample['text'], prediction)
accuracy = (1 - error_rate) * 100

print("\n" + "="*80)
print("METRICS")
print("="*80)
print(f"Character Error Rate (CER): {error_rate:.2%}")
print(f"Character Accuracy: {accuracy:.2f}%")
print(f"Ground Truth Length: {len(test_sample['text'])} chars")
print(f"Predicted Length: {len(prediction)} chars")
print("="*80)

In [ ]:
import os
import json
import torch
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
import pickle

from jiwer import cer, wer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import nltk
import warnings
warnings.filterwarnings('ignore')

try:
    nltk.data.find('wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)
    nltk.download('punkt', quiet=True)
    nltk.download('omw-1.4', quiet=True)

In [ ]:
# Load model from scratch
print("="*80)
print("LOADING MODEL FROM SCRATCH")
print("="*80)

from unsloth import FastVisionModel
import torch
from transformers import AutoModel
from datasets import load_dataset
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = '0'

print("\nLoading base DeepSeek-OCR model...")
model, tokenizer = FastVisionModel.from_pretrained(
    "./sinhala_deepseek_ocr_lora",
    load_in_4bit=True,
    auto_model=AutoModel,
    trust_remote_code=True,
    unsloth_force_compile=True,
    use_gradient_checkpointing="unsloth",
)

In [ ]:
print("\nAdding LoRA adapter configuration...")
model = FastVisionModel.get_peft_model(
    model,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
)

In [ ]:
from datasets import load_dataset
from huggingface_hub import login
import pandas as pd
from PIL import Image

login()

In [ ]:
print("\n" + "="*80)
print("LOADING TEST DATASET")
print("="*80)

print("\nLoading dataset from Hugging Face...")
dataset = load_dataset("avishadilhara/sinhala-ocr-lk-acts-1010")

print(f"Dataset loaded!")
print(f"   Test samples: {len(dataset['test'])}")

In [ ]:
CHECKPOINT_INTERVAL = 10
CHECKPOINT_FILE = "./test_results/checkpoint.pkl"

In [ ]:
def calculate_anls(ground_truth, prediction, threshold=0.5):
    """Calculate Average Normalized Levenshtein Similarity (ANLS)"""
    try:
        from Levenshtein import distance as levenshtein_distance

        if len(ground_truth) == 0 and len(prediction) == 0:
            return 1.0
        if len(ground_truth) == 0 or len(prediction) == 0:
            return 0.0

        edit_distance = levenshtein_distance(ground_truth, prediction)
        max_len = max(len(ground_truth), len(prediction))
        normalized_distance = edit_distance / max_len

        if normalized_distance < threshold:
            anls = 1 - normalized_distance
        else:
            anls = 0.0

        return anls
    except ImportError:
        from jiwer import cer
        return 1 - cer(ground_truth, prediction)


def calculate_all_metrics(ground_truth, prediction):
    """Calculate all metrics: CER, WER, BLEU, ANLS, METEOR"""
    metrics = {}

    try:
        metrics['CER'] = cer(ground_truth, prediction)
    except:
        metrics['CER'] = 1.0

    try:
        metrics['WER'] = wer(ground_truth, prediction)
    except:
        metrics['WER'] = 1.0

    try:
        reference = [list(ground_truth)]
        hypothesis = list(prediction)
        smoothing = SmoothingFunction().method1
        metrics['BLEU'] = sentence_bleu(reference, hypothesis,
                                        smoothing_function=smoothing)
    except:
        metrics['BLEU'] = 0.0

    try:
        metrics['ANLS'] = calculate_anls(ground_truth, prediction)
    except:
        metrics['ANLS'] = 0.0

    try:
        reference_tokens = list(ground_truth)
        hypothesis_tokens = list(prediction)
        metrics['METEOR'] = meteor_score([reference_tokens], hypothesis_tokens)
    except:
        metrics['METEOR'] = 0.0

    metrics['Char_Accuracy'] = (1 - metrics['CER']) * 100

    return metrics

In [ ]:
def save_checkpoint(all_results, year_results, last_processed_idx, output_dir):
    """Save checkpoint to resume later"""
    checkpoint_data = {
        'all_results': all_results,
        'year_results': dict(year_results),
        'last_processed_idx': last_processed_idx
    }

    checkpoint_path = os.path.join(output_dir, "checkpoint.pkl")
    with open(checkpoint_path, 'wb') as f:
        pickle.dump(checkpoint_data, f)


def load_checkpoint(output_dir):
    """Load checkpoint if exists"""
    checkpoint_path = os.path.join(output_dir, "checkpoint.pkl")

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'rb') as f:
            checkpoint_data = pickle.load(f)
        return checkpoint_data
    return None

In [ ]:
def test_entire_dataset(resume=True):
    """Test model on entire test dataset with checkpoint support"""

    output_dir = "./test_results"
    inference_dir = os.path.join(output_dir, "inference_outputs")
    os.makedirs(inference_dir, exist_ok=True)
    os.makedirs(os.path.join(inference_dir, "images"), exist_ok=True)

    all_results = []
    year_results = defaultdict(list)
    start_idx = 0

    if resume:
        checkpoint = load_checkpoint(output_dir)
        if checkpoint:
            all_results = checkpoint['all_results']
            year_results = defaultdict(list, checkpoint['year_results'])
            start_idx = checkpoint['last_processed_idx'] + 1

            print("="*80)
            print("CHECKPOINT FOUND - RESUMING FROM PREVIOUS RUN")
            print("="*80)
            print(f"Already processed: {len(all_results)} samples")
            print(f"Resuming from sample: {start_idx}")
            print("="*80 + "\n")

    test_dataset = dataset['test']
    total_samples = len(test_dataset)

    if start_idx == 0:
        print("\n" + "="*80)
        print("TESTING MODEL ON ENTIRE TEST DATASET")
        print("="*80)
        print(f"\nTotal samples to test: {total_samples}")
        print(f"Output directory: {inference_dir}")
        print(f"Checkpoint interval: Every {CHECKPOINT_INTERVAL} samples")
        print("="*80 + "\n")

    samples_to_process = range(start_idx, total_samples)

    with tqdm(total=total_samples, initial=start_idx, desc="Testing",
              unit="sample", ncols=100) as pbar:

        for idx in samples_to_process:
            try:
                sample = test_dataset[idx]

                ground_truth = sample['text']
                year = sample.get('year', 'Unknown')
                image = sample['image']

                img_filename = f"sample_{idx:04d}.jpg"
                temp_img_path = os.path.join(inference_dir, "images", img_filename)
                image.save(temp_img_path)

                _ = model.infer(
                    tokenizer,
                    prompt="<image> OCR.",
                    image_file=temp_img_path,
                    output_path=inference_dir,
                    base_size=1024,
                    image_size=640,
                    crop_mode=True,
                    save_results=True
                )

                prediction = ""
                mmd_file = os.path.join(inference_dir, "result.mmd")

                if os.path.exists(mmd_file):
                    with open(mmd_file, 'r', encoding='utf-8') as f:
                        prediction = f.read().strip()

                    saved_mmd = os.path.join(inference_dir, f"result_{idx:04d}.mmd")
                    os.rename(mmd_file, saved_mmd)
                else:
                    for file in os.listdir(inference_dir):
                        if file.endswith('.mmd'):
                            mmd_path = os.path.join(inference_dir, file)
                            with open(mmd_path, 'r', encoding='utf-8') as f:
                                prediction = f.read().strip()
                            saved_mmd = os.path.join(inference_dir, f"result_{idx:04d}.mmd")
                            os.rename(mmd_path, saved_mmd)
                            break

                metrics = calculate_all_metrics(ground_truth, prediction)

                result = {
                    'sample_idx': idx,
                    'year': year,
                    'ground_truth_length': len(ground_truth),
                    'prediction_length': len(prediction),
                    **metrics
                }

                all_results.append(result)
                year_results[year].append(result)

                pbar.set_postfix({
                    'CER': f"{metrics['CER']:.3f}",
                    'Acc': f"{metrics['Char_Accuracy']:.1f}%"
                })
                pbar.update(1)

                if (idx + 1) % CHECKPOINT_INTERVAL == 0:
                    save_checkpoint(all_results, year_results, idx, output_dir)
                    pbar.write(f"Checkpoint saved at sample {idx + 1}")

            except KeyboardInterrupt:
                print("\n\nTesting interrupted by user!")
                print("Saving checkpoint...")
                save_checkpoint(all_results, year_results, idx - 1, output_dir)
                print(f"Progress saved. Resume by running the code again.")
                print(f"   Processed: {len(all_results)} / {total_samples} samples\n")
                raise

            except Exception as e:
                pbar.write(f"Error at sample {idx}: {str(e)[:50]}...")

                all_results.append({
                    'sample_idx': idx,
                    'year': year if 'year' in locals() else 'Unknown',
                    'CER': 1.0,
                    'WER': 1.0,
                    'BLEU': 0.0,
                    'ANLS': 0.0,
                    'METEOR': 0.0,
                    'Char_Accuracy': 0.0,
                    'ground_truth_length': 0,
                    'prediction_length': 0
                })
                pbar.update(1)
                continue

    save_checkpoint(all_results, year_results, total_samples - 1, output_dir)
    print("\nAll samples processed!")

    return all_results, year_results

In [ ]:
# ========================================
# STEP 5: DISPLAY RESULTS
# ========================================

def display_results(all_results, year_results):
    """Display comprehensive results"""
    print("\n" + "="*80)
    print(" TEST RESULTS")
    print("="*80)

    # Convert to DataFrame
    df = pd.DataFrame(all_results)

    # ========================================
    # 1. OVERALL AVERAGE METRICS
    # ========================================
    print("\n" + "="*80)
    print("🎯 OVERALL AVERAGE METRICS")
    print("="*80)

    metrics_to_show = ['CER', 'WER', 'BLEU', 'ANLS', 'METEOR', 'Char_Accuracy']

    for metric in metrics_to_show:
        if metric in df.columns:
            avg_value = df[metric].mean()

            if metric in ['CER', 'WER']:
                direction = "↓"
                quality = "Lower is better"
            else:
                direction = "↑"
                quality = "Higher is better"

            print(f"{metric:15} {direction}: {avg_value:7.4f}  ({quality})")

    print(f"\n{'Total Samples':15}: {len(df)}")

    # ========================================
    # 2. YEAR-WISE AVERAGE METRICS
    # ========================================
    print("\n" + "="*80)
    print(" YEAR-WISE AVERAGE METRICS")
    print("="*80)

    # Group by year
    year_groups = df.groupby('year')

    # Create year summary table
    year_summary = []

    for year, group in sorted(year_groups):
        year_data = {
            'Year': year,
            'Samples': len(group),
            'CER↓': f"{group['CER'].mean():.4f}",
            'WER↓': f"{group['WER'].mean():.4f}",
            'BLEU↑': f"{group['BLEU'].mean():.4f}",
            'ANLS↑': f"{group['ANLS'].mean():.4f}",
            'METEOR↑': f"{group['METEOR'].mean():.4f}",
            'Accuracy%': f"{group['Char_Accuracy'].mean():.2f}%"
        }
        year_summary.append(year_data)

    year_df = pd.DataFrame(year_summary)

    # Display year summary table
    print("\n" + year_df.to_string(index=False))

    # ========================================
    # 3. INDIVIDUAL SAMPLE RESULTS (First 10)
    # ========================================
    print("\n" + "="*80)
    print("🔍 INDIVIDUAL SAMPLE RESULTS (First 10 samples)")
    print("="*80)

    display_cols = ['sample_idx', 'year', 'CER', 'WER', 'BLEU', 'ANLS', 'METEOR', 'Char_Accuracy']
    print("\n" + df[display_cols].head(10).to_string(index=False))

    # ========================================
    # 4. BEST AND WORST PERFORMING SAMPLES
    # ========================================
    print("\n" + "="*80)
    print(" BEST PERFORMING SAMPLES (Top 5 by Character Accuracy)")
    print("="*80)

    best_samples = df.nlargest(5, 'Char_Accuracy')[display_cols]
    print("\n" + best_samples.to_string(index=False))

    print("\n" + "="*80)
    print(" WORST PERFORMING SAMPLES (Bottom 5 by Character Accuracy)")
    print("="*80)

    worst_samples = df.nsmallest(5, 'Char_Accuracy')[display_cols]
    print("\n" + worst_samples.to_string(index=False))

    # ========================================
    # 5. SAVE RESULTS TO CSV
    # ========================================
    output_csv = "./test_results/test_metrics.csv"
    df.to_csv(output_csv, index=False, encoding='utf-8')
    print(f"\n Full results saved to: {output_csv}")

    year_csv = "./test_results/year_wise_metrics.csv"
    year_df.to_csv(year_csv, index=False)
    print(f" Year-wise results saved to: {year_csv}")

    # ========================================
    # 6. SUMMARY STATISTICS
    # ========================================
    print("\n" + "="*80)
    print(" SUMMARY STATISTICS")
    print("="*80)

    print(f"\nSamples with >90% accuracy: {len(df[df['Char_Accuracy'] > 90])}")
    print(f"Samples with >80% accuracy: {len(df[df['Char_Accuracy'] > 80])}")
    print(f"Samples with <50% accuracy: {len(df[df['Char_Accuracy'] < 50])}")

    print(f"\nMedian Character Accuracy: {df['Char_Accuracy'].median():.2f}%")
    print(f"Std Dev Character Accuracy: {df['Char_Accuracy'].std():.2f}%")

    return df, year_df

In [ ]:
if __name__ == "__main__":
    print("\n" + "="*80)
    print("STARTING COMPREHENSIVE MODEL TESTING")
    print("="*80)

    try:
        all_results, year_results = test_entire_dataset(resume=True)

        results_df, year_df = display_results(all_results, year_results)

        print("\n" + "="*80)
        print("TESTING COMPLETED!")
        print("="*80)
        print(f"\nCheck ./test_results/ directory for:")
        print(f"   - test_metrics.csv (all samples)")
        print(f"   - year_wise_metrics.csv (year summary)")
        print(f"   - inference_outputs/ (model predictions)")
        print(f"   - checkpoint.pkl (resume checkpoint)")

    except KeyboardInterrupt:
        print("\n\nTesting paused. Run again to resume from checkpoint.")